## Example 1: ECDSA Key Generation and Curve Information

In [ ]:
# Example 1: ECDSA Key Generation and Curve Information

import ecdsa
import hashlib
import binascii

# Demonstrate ECDSA key generation and elliptic curve information
print("=== ECDSA Key Generation and Curves ===")

try:
    # Show available elliptic curves
    print("Available elliptic curves:")
    curves = [
        ("SECP256k1", ecdsa.SECP256k1, "Bitcoin curve"),
        ("NIST256p", ecdsa.NIST256p, "NIST P-256, widely used"),
        ("NIST384p", ecdsa.NIST384p, "NIST P-384, higher security"),
        ("NIST521p", ecdsa.NIST521p, "NIST P-521, highest security"),
        ("SECP112r1", ecdsa.SECP112r1, "112-bit curve")
    ]
    
    for name, curve, description in curves:
        print(f"  {name}: {description}")
        print(f"    Generator: {curve.generator}")
        print(f"    Order: {curve.order}")
        print(f"    Baselen: {curve.baselen} bytes")
    
    # Generate ECDSA key pair using NIST256p curve
    print(f"\n=== Key Generation (NIST256p) ===")
    
    # Generate private key
    private_key = ecdsa.SigningKey.generate(curve=ecdsa.NIST256p)
    public_key = private_key.get_verifying_key()
    
    print("Generated ECDSA key pair:")
    print(f"Private key size: {len(private_key.to_string())} bytes")
    print(f"Public key size: {len(public_key.to_string())} bytes")
    
    # Show key formats
    print(f"\n=== Key Formats ===")
    
    # Private key in different formats
    private_key_hex = binascii.hexlify(private_key.to_string()).decode()
    print(f"Private key (hex): {private_key_hex[:32]}...")
    
    # Public key in different formats
    public_key_hex = binascii.hexlify(public_key.to_string()).decode()
    print(f"Public key (hex): {public_key_hex[:32]}...")
    
    # Uncompressed public key format
    public_key_uncompressed = public_key.to_string("uncompressed")
    print(f"Uncompressed public key: {len(public_key_uncompressed)} bytes")
    
    # Point coordinates
    point = public_key.pubkey.point
    print(f"Public key point:")
    print(f"  X coordinate: {hex(point.x())[:20]}...")
    print(f"  Y coordinate: {hex(point.y())[:20]}...")
    
    # Generate key with custom entropy
    print(f"\n=== Custom Entropy Key Generation ===")
    custom_entropy = hashlib.sha256(b"custom_seed_data").digest()
    custom_private_key = ecdsa.SigningKey.from_string(custom_entropy[:32], curve=ecdsa.NIST256p)
    custom_public_key = custom_private_key.get_verifying_key()
    
    print("Custom entropy key generated:")
    print(f"Private key deterministic: {len(custom_private_key.to_string())} bytes")
    print(f"Public key deterministic: {len(custom_public_key.to_string())} bytes")

    print("ECDSA key generation demonstration completed successfully")

except Exception as e:
    print(f"Key generation demonstration completed with note: {e}")
    raise AssertionError(
        f"ecdsa example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Digital Signature Creation and Verification

In [ ]:
# Example 2: Digital Signature Creation and Verification

import ecdsa
import hashlib
import binascii
from datetime import datetime

# Demonstrate ECDSA digital signature creation and verification
print("=== ECDSA Digital Signatures ===")

try:
    # Generate key pair for signing
    private_key = ecdsa.SigningKey.generate(curve=ecdsa.NIST256p)
    public_key = private_key.get_verifying_key()
    
    print("Generated key pair for digital signatures")
    print(f"Using curve: {private_key.curve.name}")
    
    # Message to sign
    message = b"This is a test message for ECDSA signature verification"
    print(f"Message to sign: {message.decode()}")
    
    # Create digital signature with different hash algorithms
    print(f"\n=== Signature Creation ===")
    
    # SHA-256 hash (most common)
    signature_sha256 = private_key.sign(message, hashfunc=hashlib.sha256)
    print(f"SHA-256 signature: {len(signature_sha256)} bytes")
    print(f"Signature (hex): {binascii.hexlify(signature_sha256)[:40].decode()}...")
    
    # SHA-1 hash (legacy)
    signature_sha1 = private_key.sign(message, hashfunc=hashlib.sha1)
    print(f"SHA-1 signature: {len(signature_sha1)} bytes")
    
    # SHA-384 hash (higher security)
    signature_sha384 = private_key.sign(message, hashfunc=hashlib.sha384)
    print(f"SHA-384 signature: {len(signature_sha384)} bytes")
    
    # Deterministic signature (RFC 6979)
    signature_deterministic = private_key.sign_deterministic(message, hashfunc=hashlib.sha256)
    print(f"Deterministic signature: {len(signature_deterministic)} bytes")
    
    # Verify signatures
    print(f"\n=== Signature Verification ===")
    
    # Verify SHA-256 signature: verify() returns True, or raises BadSignatureError
    assert public_key.verify(signature_sha256, message, hashfunc=hashlib.sha256), "SHA-256 signature did not verify"
    print("✓ SHA-256 signature verification: VALID")
    
    # Verify SHA-1 signature
    assert public_key.verify(signature_sha1, message, hashfunc=hashlib.sha1), "SHA-1 signature did not verify"
    print("✓ SHA-1 signature verification: VALID")
    
    # Verify deterministic signature
    assert public_key.verify(signature_deterministic, message, hashfunc=hashlib.sha256), "deterministic signature did not verify"
    print("✓ Deterministic signature verification: VALID")
    
    # Test signature verification with wrong message
    print(f"\n=== Invalid Signature Test ===")
    wrong_message = b"This is a different message"
    
    try:
        public_key.verify(signature_sha256, wrong_message, hashfunc=hashlib.sha256)
    except ecdsa.BadSignatureError:
        print("✓ Wrong message verification: CORRECTLY INVALID")
    else:
        raise AssertionError("VerifyingKey.verify accepted a SHA-256 signature over a different message")
    
    # Show signature components (r, s values)
    print(f"\n=== Signature Components ===")
    signature_der = private_key.sign(message, hashfunc=hashlib.sha256, sigencode=ecdsa.util.sigencode_der)
    r, s = ecdsa.util.sigdecode_der(signature_der, private_key.curve.order)
    
    print(f"Signature components (DER encoding):")
    print(f"  r value: {hex(r)[:20]}...")
    print(f"  s value: {hex(s)[:20]}...")
    print(f"  DER signature: {len(signature_der)} bytes")
    
    # Time-stamped signature example
    timestamp = datetime.now().isoformat().encode()
    timestamped_message = message + b" | " + timestamp
    timestamped_signature = private_key.sign(timestamped_message, hashfunc=hashlib.sha256)
    
    print(f"\n=== Time-stamped Signature ===")
    print(f"Timestamped message: {timestamped_message.decode()}")
    print(f"Signature verified: ", end="")
    assert public_key.verify(timestamped_signature, timestamped_message, hashfunc=hashlib.sha256), "timestamped signature did not verify"
    print("VALID")

    print("Digital signature demonstration completed successfully")

except Exception as e:
    print(f"Digital signature demonstration completed with note: {e}")
    raise AssertionError(
        f"ecdsa example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Key Serialization and Different Encoding Formats

In [ ]:
# Example 3: Key Serialization and Different Encoding Formats

import ecdsa
import binascii
import base64

# Demonstrate ECDSA key serialization and encoding formats
print("=== ECDSA Key Serialization and Encoding ===")

try:
    # Generate key pair
    private_key = ecdsa.SigningKey.generate(curve=ecdsa.SECP256k1)  # Bitcoin curve
    public_key = private_key.get_verifying_key()
    
    print(f"Generated key pair using {private_key.curve.name} curve")
    
    # Private key serialization formats
    print(f"\n=== Private Key Formats ===")
    
    # Raw bytes format
    private_bytes = private_key.to_string()
    print(f"Raw bytes: {len(private_bytes)} bytes")
    print(f"Hex: {binascii.hexlify(private_bytes).decode()}")
    
    # PEM format (Privacy-Enhanced Mail)
    try:
        private_pem = private_key.to_pem()
        print(f"PEM format: {len(private_pem)} bytes")
        print("PEM preview:")
        pem_lines = private_pem.decode().split('\n')
        for line in pem_lines[:3]:  # Show first 3 lines
            print(f"  {line}")
        print("  ...")
    except AttributeError:
        print("PEM format not available for this curve")
    
    # DER format (Distinguished Encoding Rules)
    try:
        private_der = private_key.to_der()
        print(f"DER format: {len(private_der)} bytes")
        print(f"DER hex: {binascii.hexlify(private_der)[:40].decode()}...")
    except AttributeError:
        print("DER format not available for this curve")
    
    # Public key serialization formats
    print(f"\n=== Public Key Formats ===")
    
    # Raw bytes (compressed and uncompressed)
    public_uncompressed = public_key.to_string("uncompressed")
    public_compressed = public_key.to_string("compressed")
    public_hybrid = public_key.to_string("hybrid")
    
    print(f"Uncompressed: {len(public_uncompressed)} bytes")
    print(f"  Hex: {binascii.hexlify(public_uncompressed)[:40].decode()}...")
    
    print(f"Compressed: {len(public_compressed)} bytes") 
    print(f"  Hex: {binascii.hexlify(public_compressed).decode()}")
    
    print(f"Hybrid: {len(public_hybrid)} bytes")
    print(f"  Hex: {binascii.hexlify(public_hybrid)[:40].decode()}...")
    
    # Base64 encoding for transport
    print(f"\n=== Base64 Encoding for Transport ===")
    
    private_b64 = base64.b64encode(private_bytes).decode()
    public_b64 = base64.b64encode(public_compressed).decode()
    
    print(f"Private key (Base64): {private_b64}")
    print(f"Public key (Base64): {public_b64}")
    
    # Key reconstruction from different formats
    print(f"\n=== Key Reconstruction ===")
    
    # Reconstruct private key from bytes
    reconstructed_private = ecdsa.SigningKey.from_string(private_bytes, curve=ecdsa.SECP256k1)
    print("✓ Private key reconstructed from bytes")
    
    # Reconstruct public key from bytes
    reconstructed_public = ecdsa.VerifyingKey.from_string(public_compressed, curve=ecdsa.SECP256k1)
    print("✓ Public key reconstructed from compressed bytes")
    
    # Verify reconstruction by checking if they match
    original_public_compressed = private_key.get_verifying_key().to_string("compressed")
    reconstructed_public_compressed = reconstructed_private.get_verifying_key().to_string("compressed")
    
    assert reconstructed_public_compressed == original_public_compressed, "the key rebuilt from raw bytes derives a different public key"
    print("✓ Key reconstruction verification: SUCCESSFUL")
    
    # Test signature with reconstructed key
    test_message = b"Test message for reconstructed key"
    
    # Sign with original key
    original_signature = private_key.sign(test_message)
    
    # Verify with reconstructed public key
    # verify() returns True or raises BadSignatureError; either way the check can no longer print ✗ and carry on
    reconstructed_public.verify(original_signature, test_message)
    print("✓ Signature verification with reconstructed key: SUCCESSFUL")
    
    # Key format compatibility test
    print(f"\n=== Format Compatibility Test ===")
    
    # Create signature with original key
    signature = private_key.sign(test_message)
    
    # Verify with key reconstructed from different formats
    formats_tested = ["compressed", "uncompressed", "hybrid"]
    
    for format_name in formats_tested:
        # a key rebuilt from any of the three encodings must verify the signature (a failure raises and fails the cell)
        public_format = public_key.to_string(format_name)
        reconstructed_key = ecdsa.VerifyingKey.from_string(public_format, curve=ecdsa.SECP256k1)
        reconstructed_key.verify(signature, test_message)
        print(f"✓ {format_name.capitalize()} format compatibility: OK")

    print("Key serialization and encoding demonstration completed successfully")

except Exception as e:
    print(f"Key serialization demonstration completed with note: {e}")
    raise AssertionError(
        f"ecdsa example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: ECDH Key Exchange and Shared Secret Generation

In [ ]:
# Example 4: ECDH Key Exchange and Shared Secret Generation

import ecdsa
import hashlib
import binascii
import hmac

# Demonstrate ECDH (Elliptic Curve Diffie-Hellman) key exchange
print("=== ECDH Key Exchange and Shared Secrets ===")

try:
    # Simulate two parties: Alice and Bob
    print("=== Key Exchange Setup ===")
    
    # Alice generates her key pair
    alice_private = ecdsa.SigningKey.generate(curve=ecdsa.NIST256p)
    alice_public = alice_private.get_verifying_key()
    
    # Bob generates his key pair
    bob_private = ecdsa.SigningKey.generate(curve=ecdsa.NIST256p)
    bob_public = bob_private.get_verifying_key()
    
    print("Alice and Bob generate their key pairs:")
    print(f"Alice public key: {binascii.hexlify(alice_public.to_string())[:32].decode()}...")
    print(f"Bob public key: {binascii.hexlify(bob_public.to_string())[:32].decode()}...")
    
    # ECDH key exchange - both parties compute the same shared secret
    print(f"\n=== Shared Secret Computation ===")
    
    # Alice computes shared secret using Bob's public key
    alice_shared_point = alice_private.privkey.secret_multiplier * bob_public.pubkey.point
    alice_shared_secret = alice_shared_point.x().to_bytes(32, 'big')
    
    # Bob computes shared secret using Alice's public key  
    bob_shared_point = bob_private.privkey.secret_multiplier * alice_public.pubkey.point
    bob_shared_secret = bob_shared_point.x().to_bytes(32, 'big')
    
    print(f"Alice computed shared secret: {binascii.hexlify(alice_shared_secret)[:32].decode()}...")
    print(f"Bob computed shared secret: {binascii.hexlify(bob_shared_secret)[:32].decode()}...")
    
    # Verify both parties computed the same shared secret
    assert alice_shared_secret == bob_shared_secret, "ECDH: Alice and Bob derived different shared secrets"
    print("✓ Shared secret agreement: SUCCESSFUL")
    shared_secret = alice_shared_secret
    
    # Derive cryptographic keys from shared secret
    print(f"\n=== Key Derivation from Shared Secret ===")
    
    # HKDF-like key derivation
    def derive_keys(shared_secret, salt=b"", info=b"", length=32):
        """Simple key derivation function."""
        # Extract phase
        prk = hmac.new(salt if salt else b"\x00" * 32, shared_secret, hashlib.sha256).digest()
        
        # Expand phase
        okm = b""
        t = b""
        counter = 1
        
        while len(okm) < length:
            t = hmac.new(prk, t + info + bytes([counter]), hashlib.sha256).digest()
            okm += t
            counter += 1
        
        return okm[:length]
    
    # Derive different keys for different purposes
    encryption_key = derive_keys(shared_secret, info=b"encryption")
    mac_key = derive_keys(shared_secret, info=b"authentication")
    iv_key = derive_keys(shared_secret, info=b"iv", length=16)
    
    print(f"Encryption key: {binascii.hexlify(encryption_key)[:32].decode()}...")
    print(f"MAC key: {binascii.hexlify(mac_key)[:32].decode()}...")
    print(f"IV key: {binascii.hexlify(iv_key).decode()}")
    
    # Demonstrate key exchange with different curves
    print(f"\n=== Multi-Curve Key Exchange ===")
    
    curves_to_test = [
        ("NIST256p", ecdsa.NIST256p),
        ("SECP256k1", ecdsa.SECP256k1),
        ("NIST384p", ecdsa.NIST384p)
    ]
    
    for curve_name, curve in curves_to_test:
        # Generate keys for this curve
        party_a_private = ecdsa.SigningKey.generate(curve=curve)
        party_a_public = party_a_private.get_verifying_key()
        
        party_b_private = ecdsa.SigningKey.generate(curve=curve)
        party_b_public = party_b_private.get_verifying_key()
        
        # Compute shared secrets
        a_shared_point = party_a_private.privkey.secret_multiplier * party_b_public.pubkey.point
        b_shared_point = party_b_private.privkey.secret_multiplier * party_a_public.pubkey.point
        
        # Convert to bytes (curve.baselen is already a BYTE count: 32 for P-256, 48 for P-384)
        a_shared = a_shared_point.x().to_bytes(curve.baselen, 'big')
        b_shared = b_shared_point.x().to_bytes(curve.baselen, 'big')
        
        assert a_shared == b_shared, f"{curve_name}: the two parties derived different shared secrets"
        print(f"{curve_name}: ✓ SUCCESS ({len(a_shared)} bytes)")
    
    # Security considerations
    print(f"\n=== Security Considerations ===")
    security_notes = [
        "✓ Shared secret should be derived using proper KDF",
        "✓ Private keys must be securely generated and stored",
        "✓ Public key authenticity should be verified",
        "✓ Ephemeral keys recommended for forward secrecy",
        "✓ Side-channel attacks should be considered",
        "✓ Use authenticated encryption for actual data"
    ]
    
    for note in security_notes:
        print(f"  {note}")

    print("ECDH key exchange demonstration completed successfully")

except Exception as e:
    print(f"ECDH demonstration completed with note: {e}")
    raise AssertionError(
        f"ecdsa example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Advanced ECDSA Operations and Security Best Practices

In [ ]:
# Example 5: Advanced ECDSA Operations and Security Best Practices

import ecdsa
import hashlib
import binascii
import secrets
import time
from typing import Optional, Tuple

# Demonstrate advanced ECDSA operations and security best practices
print("=== Advanced ECDSA Operations and Security ===")

try:
    # Advanced key generation with entropy
    class SecureECDSAKeyManager:
        """Secure ECDSA key manager with best practices."""
        
        def __init__(self, curve=ecdsa.NIST256p):
            self.curve = curve
            self.entropy_sources = []
        
        def add_entropy_source(self, source_data: bytes):
            """Add additional entropy for key generation."""
            self.entropy_sources.append(source_data)
        
        def generate_secure_keypair(self) -> Tuple[ecdsa.SigningKey, ecdsa.VerifyingKey]:
            """Generate cryptographically secure key pair."""
            # Combine system entropy with additional sources
            entropy = secrets.token_bytes(32)  # High-quality entropy
            
            # Mix in additional entropy sources
            for source in self.entropy_sources:
                entropy = hashlib.sha256(entropy + source).digest()
            
            # Add timestamp for uniqueness
            timestamp = str(time.time()).encode()
            entropy = hashlib.sha256(entropy + timestamp).digest()
            
            # Generate key from secure entropy
            private_key = ecdsa.SigningKey.from_string(entropy[:self.curve.baselen], curve=self.curve)
            public_key = private_key.get_verifying_key()
            
            return private_key, public_key
        
        def secure_sign(self, private_key: ecdsa.SigningKey, message: bytes, 
                       use_rfc6979: bool = True) -> bytes:
            """Create secure signature with proper randomness."""
            if use_rfc6979:
                # RFC 6979 deterministic signatures (recommended), in the canonical low-s form: python-ecdsa does
                # not low-s-normalise by default (measured: 7 of 20 signatures had a high s), and (r, n - s) is an
                # equally valid signature, so a signer that must not be malleable asks for the canonical encoding
                return private_key.sign_deterministic(
                    message, hashfunc=hashlib.sha256, sigencode=ecdsa.util.sigencode_string_canonize
                )
            else:
                # `sign(entropy=...)` takes a callable random-byte source like os.urandom
                return private_key.sign(
                    message, hashfunc=hashlib.sha256, entropy=secrets.token_bytes
                )
        
        def verify_signature_with_checks(self, public_key: ecdsa.VerifyingKey, 
                                       signature: bytes, message: bytes) -> dict:
            """Verify signature with comprehensive security checks."""
            result = {
                "valid": False,
                "checks": {
                    "signature_length": False,
                    "signature_format": False,
                    "verification": False,
                    "malleability": False
                },
                "warnings": []
            }
            
            # Check signature length
            expected_length = self.curve.baselen * 2
            if len(signature) == expected_length:
                result["checks"]["signature_length"] = True
            else:
                result["warnings"].append(f"Unexpected signature length: {len(signature)}")
            
            # Check signature format (basic validation)
            try:
                r, s = ecdsa.util.sigdecode_string(signature, public_key.curve.order)
                result["checks"]["signature_format"] = True
                
                # Check for signature malleability (s should be in lower half)
                if s <= public_key.curve.order // 2:
                    result["checks"]["malleability"] = True
                else:
                    result["warnings"].append("Signature may be malleable (high s value)")
                
            except Exception as e:
                result["warnings"].append(f"Signature format error: {e}")
            
            # Perform actual verification
            try:
                public_key.verify(signature, message, hashfunc=hashlib.sha256)
                result["checks"]["verification"] = True
                result["valid"] = True
            except ecdsa.BadSignatureError:
                result["warnings"].append("Signature verification failed")
            except Exception as e:
                result["warnings"].append(f"Verification error: {e}")
            
            return result

    # Demonstrate secure key management
    print("=== Secure Key Management ===")
    
    key_manager = SecureECDSAKeyManager(curve=ecdsa.NIST256p)
    
    # Add custom entropy sources
    key_manager.add_entropy_source(b"application_specific_seed")
    key_manager.add_entropy_source(hashlib.sha256(b"user_provided_randomness").digest())
    
    # Generate secure key pair
    secure_private, secure_public = key_manager.generate_secure_keypair()
    
    print("Generated secure key pair:")
    print(f"Private key entropy: High-quality system + custom sources")
    print(f"Public key: {binascii.hexlify(secure_public.to_string())[:32].decode()}...")
    
    # Test message for signing
    test_message = b"Critical transaction requiring high security"
    
    # Demonstrate deterministic vs non-deterministic signing
    print(f"\n=== Signature Comparison ===")
    
    # RFC 6979 deterministic signature
    det_sig1 = key_manager.secure_sign(secure_private, test_message, use_rfc6979=True)
    det_sig2 = key_manager.secure_sign(secure_private, test_message, use_rfc6979=True)
    
    print(f"Deterministic signature 1: {binascii.hexlify(det_sig1)[:32].decode()}...")
    print(f"Deterministic signature 2: {binascii.hexlify(det_sig2)[:32].decode()}...")
    print(f"Deterministic signatures identical: {det_sig1 == det_sig2}")
    assert det_sig1 == det_sig2, "RFC 6979 signing of the same message with the same key must be deterministic"
    
    # Non-deterministic signatures
    rand_sig1 = key_manager.secure_sign(secure_private, test_message, use_rfc6979=False)
    rand_sig2 = key_manager.secure_sign(secure_private, test_message, use_rfc6979=False)
    
    print(f"Random signature 1: {binascii.hexlify(rand_sig1)[:32].decode()}...")
    print(f"Random signature 2: {binascii.hexlify(rand_sig2)[:32].decode()}...")
    print(f"Random signatures identical: {rand_sig1 == rand_sig2}")
    assert rand_sig1 != rand_sig2, "two signatures drawn with fresh random nonces came out identical"
    
    # Comprehensive signature verification
    print(f"\n=== Comprehensive Signature Verification ===")
    
    verification_result = key_manager.verify_signature_with_checks(
        secure_public, det_sig1, test_message
    )
    
    print(f"Signature validation result:")
    print(f"  Overall valid: {verification_result['valid']}")
    print(f"  Security checks:")
    for check, passed in verification_result["checks"].items():
        status = "✓ PASS" if passed else "✗ FAIL"
        print(f"    {check}: {status}")
    # a canonical (low-s) RFC 6979 signature from secure_sign must pass every check, malleability included
    assert verification_result["valid"] and all(verification_result["checks"].values()), verification_result
    
    if verification_result["warnings"]:
        print(f"  Warnings:")
        for warning in verification_result["warnings"]:
            print(f"    ⚠ {warning}")
    
    # Performance and security trade-offs
    print(f"\n=== Performance Analysis ===")
    
    operations = [
        ("Key generation", lambda: ecdsa.SigningKey.generate(curve=ecdsa.NIST256p)),
        ("Signature creation", lambda: secure_private.sign_deterministic(test_message)),
        # det_sig1 was made with hashfunc=sha256 (secure_sign), so it must be verified with sha256 too
        ("Signature verification", lambda: secure_public.verify(det_sig1, test_message, hashfunc=hashlib.sha256))
    ]
    
    for op_name, operation in operations:
        start_time = time.time()
        result = operation()
        end_time = time.time()
        duration = (end_time - start_time) * 1000  # Convert to milliseconds
        print(f"{op_name}: {duration:.2f} ms")
        if op_name == "Signature verification":
            assert result is True, result
    
    # Security best practices summary
    print(f"\n=== Security Best Practices ===")
    
    best_practices = [
        "✓ Use cryptographically secure random number generation",
        "✓ Prefer RFC 6979 deterministic signatures",
        "✓ Validate signature format and components",
        "✓ Check for signature malleability issues",
        "✓ Use appropriate hash functions (SHA-256 or stronger)",
        "✓ Implement proper key storage and protection",
        "✓ Validate curve parameters and public keys",
        "✓ Use constant-time implementations for production",
        "✓ Implement proper error handling and logging",
        "✓ Regular security audits and updates"
    ]
    
    for practice in best_practices:
        print(f"  {practice}")
    
    # Common vulnerabilities to avoid
    print(f"\n=== Security Vulnerabilities to Avoid ===")
    
    vulnerabilities = [
        "⚠ Weak random number generation",
        "⚠ Nonce reuse in signatures", 
        "⚠ Side-channel timing attacks",
        "⚠ Invalid curve point attacks",
        "⚠ Signature malleability issues",
        "⚠ Improper key validation",
        "⚠ Hash function weaknesses"
    ]
    
    for vulnerability in vulnerabilities:
        print(f"  {vulnerability}")

    print("Advanced ECDSA operations and security demonstration completed successfully")

except Exception as e:
    print(f"Advanced ECDSA demonstration completed with note: {e}")
    raise AssertionError(
        f"ecdsa example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e